<!-- cabecera-entorno -->
## Antes de empezar

**Clase 4 · EDA: estadística, GroupBy y análisis univariado** — Bloque 2 · Demo. Este cuaderno se
recorre **por su cuenta**: explica cada concepto antes de usarlo, y el profesor circula por el salón
resolviendo dudas. No hay que esperar a que alguien lo dicte.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `demo.ipynb` como
`demo_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El notebook se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/HISTORICO_CONSUMO.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 4 · Demo — EDA: estadística, GroupBy y análisis univariado

> **Dónde vamos.** Tercera de las cuatro clases de **EDA**, análisis exploratorio de datos (2, 3,
> 4 y 5). Entender y limpiar y organizar ya están hechos; el verbo de hoy es **describir**: cada
> variable con su valor típico, su dispersión y su forma. El EDA no empieza hoy: empezó en la clase 2. El marco
> completo está en el cuaderno de la clase 2, sección *Qué estamos haciendo: EDA*.

**Dataset:** consumo de agua de Empocaldas, `../datos/HISTORICO_CONSUMO.csv`, publicado en
datos.gov.co.

## Cómo se usa este cuaderno

Está escrito para que usted avance solo. Cada bloque de código viene precedido de la explicación del
concepto que usa, y cada término nuevo se define la primera vez que aparece. Hay que leer antes de
ejecutar.

**Los recuadros "Para entender qué está pasando".** Esta clase no es solo de pandas: es de
estadística. Casi todo el mundo sabe calcular una media; muy poca gente sabe decir qué **es** una
media, y sin eso no se puede elegir entre ella y la mediana con criterio. Hay tres recuadros de ese
tipo —media contra mediana, desviación estándar, y qué es un objeto agrupado— que explican el
fundamento, no el comando. Están marcados y se pueden saltar si ya los tiene claros; si no, son la
parte más importante del cuaderno.

**Lo de pandas puro no se repite aquí.** Qué es una librería, qué es un DataFrame, qué es una Series,
qué es un `dtype`, qué es `NaN` y cómo se filtra con máscaras booleanas se explicó en el demo de la
clase 2. Este cuaderno lo da por sabido y le indica adónde volver si hace falta.

**El recorrido, de lo simple a lo complejo:**

| Sección | De qué va | Qué se lleva |
|---------|-----------|--------------|
| 0 y 1 | Qué mide este archivo y cómo se carga uno con formato colombiano | El dato en memoria, y bien |
| 2 | Paso 1 del marco: identificar | Saber qué tiene entre manos |
| 3 | Pasos 2 y 3: resumir y dispersar | Centro y dispersión, con criterio |
| 4 | GroupBy: split, apply, combine | Comparar entre grupos |
| 5 | Paso 4: visualizar | La forma de la distribución |
| 6 | Paso 5: detectar outliers | Decidir qué se hace con las jirafas |
| 7 y 8 | Los errores que no avisan y el marco completo | Cerrar el círculo |

### El marco univariado de 5 pasos

Es la tarjeta de la clase. Se aplica una vez aquí, y tres veces en el reto.

| Paso | Nombre | Qué se hace | Pregunta que responde |
|------|--------|-------------|------------------------|
| 1 | Identificar | Tipo de dato, no nulos, faltantes | ¿Qué es esta variable? |
| 2 | Resumir | Media, mediana, moda, razón media/mediana | ¿Dónde está el centro? |
| 3 | Dispersar | Desviación estándar, Q1, Q3, IQR, rango | ¿Qué tan regada está? |
| 4 | Visualizar | Histograma y boxplot | ¿Qué forma tiene? |
| 5 | Detectar | Regla 1.5xIQR, contar outliers, decidir | ¿Hay jirafas? ¿Qué hago con ellas? |

**Aquí no hay nada que teclear.** Todo el código está escrito y ejecutable: usted lo corre, mira la
salida y lee la explicación que está justo encima. Escribir código es el bloque 3, con el reto, y es
lo que se entrega.

**Las doce preguntas de interpretación** son lo que sí le toca a usted. No llevan código: se
responden escribiendo en español en la celda, debajo de *Tu respuesta:*. Cada una trae un bloque
plegable *"Comparar con la respuesta esperada"*. **Escriba la suya primero y ábralo después.**
Abrirlo antes no le ahorra nada: lo que se evalúa en el reto y en la sustentación es que usted sepa
mirar un número y decir qué significa, no que sepa reconocer una respuesta correcta cuando la ve.

**Si algo se rompe**, no siempre es un accidente: las secciones 1 y 7 provocan errores a propósito,
con la explicación al lado. Ese es el material más útil del cuaderno.

**Punto de control:** al final hay tres preguntas para responderse a sí mismo antes de pasar al reto.

---

## 0. El dataset, antes de tocarlo

Regla de la casa: nunca se ejecuta una línea de código sobre un dataset que no se sabe qué es.

| Campo | Valor |
|-------|-------|
| Qué mide | Consumo de agua y alcantarillado facturado por Empocaldas |
| Quién lo publica | Empresa de Obras Sanitarias de Caldas (Empocaldas S.A. E.S.P.), vía datos.gov.co |
| Tamaño | 21.816 filas x 12 columnas |
| Granularidad | Una fila = un municipio, un estrato, un mes, un año |
| Cobertura | 24 municipios de Caldas, 9 categorías de estrato, años 2015 a 2023 |

**Qué es la granularidad.** El nivel de detalle de una fila: qué representa exactamente **una** fila.
Es la primera pregunta que hay que responder de cualquier tabla, porque decide qué se puede sumar y
qué no. Aquí una fila **no** es un hogar ni una factura: es un grupo de suscriptores (los de un
estrato, en un municipio, en un mes).

### Las 12 columnas

| Columna | Qué es |
|---------|--------|
| `NIT` | NIT de la empresa. Constante en todas las filas |
| `RAZON SOCIAL` | Nombre de la empresa. Constante en todas las filas |
| `AÑO` | Año del registro (2015-2023) |
| `MES` | Mes en texto (ENERO ... DICIEMBRE) |
| `MUNICIPIO` | Uno de 24 municipios de Caldas |
| `ESTRATO` | Estrato1 a Estrato6, Comercial, Industrial, Publico / Oficial |
| `No. SUSCRIPTORES ACUEDUCTO` | Cuántos suscriptores de acueducto hay en ese grupo |
| `CONSUMO M3 ACUEDUCTO` | Consumo **total** del grupo, en metros cúbicos |
| `PROMEDIO CONSUMO ACUEDUCTO` | Consumo **por suscriptor**, en metros cúbicos |
| `No. SUSCRIPTORES ALCANTARILLADO` | Lo mismo, para alcantarillado |
| `CONSUMO M3 ALCANTARILLADO` | Lo mismo, para alcantarillado |
| `PROMEDIO CONSUMO ALCANTARILLADO` | Lo mismo, para alcantarillado |

**Guarde esta distinción, hoy decide toda la clase:** `CONSUMO M3 ACUEDUCTO` es una **suma** del
grupo. `PROMEDIO CONSUMO ACUEDUCTO` es un **cociente** por suscriptor. No miden lo mismo, y
compararlos entre grupos de tamaños distintos da respuestas opuestas. Lo va a ver con sus ojos en la
sección 4.

### Lo que este archivo tiene de sucio

Se dice **antes** de cargarlo, porque cambia la forma de cargarlo:

1. `AÑO` viene como texto con separador de miles: `"2,015"`, no `2015`.
2. Los consumos y los suscriptores usan **punto como separador de miles**: `"1.043"` son mil cuarenta
   y tres, no uno coma cero cuarenta y tres.
3. Las columnas `PROMEDIO` usan el punto como **decimal**: `"8.4"` son ocho coma cuatro. Formato
   distinto dentro del mismo archivo.
4. `NIT` y `RAZON SOCIAL` son constantes: no aportan información.
5. Hay valores faltantes repartidos en tres columnas.
6. Los nombres de columna tienen espacios y puntos, y obligan a escribir corchetes siempre.

Los puntos 1, 3, 5 y 6 se arreglan con lo de la clase 3. El punto 2 es el peligroso, y merece su
propia sección.

**Esto no es mala suerte.** Un archivo publicado por una entidad pública casi nunca llega listo para
analizar. La limpieza es entre el 60% y el 80% del trabajo real de un analista.

---

## 1. Cargar un archivo con formato colombiano

### 1.1 El error que sí avisa

`.astype(int)` convierte una columna de texto a número entero. Si un valor trae un carácter que no
pertenece a un número —una coma, por ejemplo— revienta. Y eso es lo bueno: le dice exactamente cuál
es el carácter que sobra.

La celda de abajo provoca el error a propósito y lo atrapa con `try / except` para que el cuaderno
siga corriendo. Lea el nombre del error y el mensaje.

In [ ]:
import pandas as pd

try:
    pd.Series(['2,015', '2,016']).astype(int)
except ValueError as error:
    print("Tipo de error:", type(error).__name__)
    print("Mensaje:", error)

print()
print("La cura: quitar la coma antes de convertir.")
print(pd.Series(['2,015', '2,016']).str.replace(',', '', regex=False).astype(int).tolist())

`.str` y `.astype()` vienen de la clase 2: `.str` es el accesorio que aplica operaciones de texto a
toda la columna de un golpe, y `.astype()` fuerza el tipo. Si alguno de los dos le suena a chino,
vuelva al demo de la clase 2, sección 4, antes de seguir.

### 1.2 El error que NO avisa

Este es el caro del día. `"1.043"` en este archivo significa **mil cuarenta y tres**: el punto es el
separador de miles a la colombiana. Si dejamos que pandas adivine los tipos al leer el CSV, va a ver
un punto y va a entender **uno coma cero cuarenta y tres**, porque el punto decimal es la convención
en inglés.

No hay error. No hay advertencia. El cuaderno corre entero y todos los resultados quedan mil veces
más chicos. Compruébelo.

In [ ]:
# Lectura ingenua: pandas adivina los tipos
ingenuo = pd.read_csv('../datos/HISTORICO_CONSUMO.csv')

print("Adivinando tipos:")
print("  tipo de la columna:", ingenuo['CONSUMO M3 ACUEDUCTO'].dtype)
print("  consumo maximo:    ", ingenuo['CONSUMO M3 ACUEDUCTO'].max())
print()
print("Ni un error, ni una advertencia. Y el numero esta mil veces mal.")
print("El maximo real de este dataset es 164.863 m3.")

**La defensa: `dtype=str`.** Le pide a pandas que **no adivine nada** y lo lea todo como texto.
Después convertimos a mano, columna por columna, decidiendo nosotros qué significa cada punto y cada
coma. Es más trabajo y es la única forma segura con datos colombianos.

La regla general del semestre: **cuando el formato numérico no es el estándar en inglés, se carga con
`dtype=str` y se convierte a mano.**

### 1.3 La carga buena

Lea la celda antes de ejecutarla. Cada bloque corresponde a uno de los seis problemas de la lista de
la sección 0.

Funciones que aparecen por primera vez:

- `pd.read_csv(ruta, dtype=str)` — lee el CSV sin adivinar tipos.
- `.drop(columns=[...])` — elimina columnas.
- `.rename(columns={...})` — cambia nombres de columna.
- `.dropna()` — elimina las filas que tengan al menos un valor faltante.
- `.astype(int)` / `.astype(float)` — conversión de tipo.

**Sobre la ruta.** `../../` significa "suba dos niveles desde la carpeta donde está este cuaderno".
Este cuaderno vive en `clase04/demo/`, así que dos niveles arriba es la raíz del curso, y de ahí baja
a `datasets/`. La ruta es relativa **al cuaderno**, no a la carpeta abierta en VSCode.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Que pandas muestre todas las columnas y los numeros con separador de miles
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Este cuaderno vive en clase04/demo/
df = pd.read_csv('../datos/HISTORICO_CONSUMO.csv', dtype=str)
print(f'Filas y columnas al cargar: {df.shape}')

# Problema 4: columnas constantes
df = df.drop(columns=['NIT', 'RAZON SOCIAL'])

# Problema 6: nombres con espacios y puntos
df = df.rename(columns={
    'AÑO': 'ANIO',
    'No. SUSCRIPTORES ACUEDUCTO': 'SUSCRIPTORES_ACUEDUCTO',
    'CONSUMO M3 ACUEDUCTO': 'CONSUMO_ACUEDUCTO',
    'PROMEDIO CONSUMO ACUEDUCTO': 'PROMEDIO_ACUEDUCTO',
    'No. SUSCRIPTORES ALCANTARILLADO': 'SUSCRIPTORES_ALCANTARILLADO',
    'CONSUMO M3 ALCANTARILLADO': 'CONSUMO_ALCANTARILLADO',
    'PROMEDIO CONSUMO ALCANTARILLADO': 'PROMEDIO_ALCANTARILLADO'
})

# Problema 5: valores faltantes
filas_antes = len(df)
df = df.dropna()
print(f'Filas eliminadas por valores faltantes: {filas_antes - len(df)}')

# Problema 1: el anio viene como "2,015"
df['ANIO'] = df['ANIO'].str.replace(',', '', regex=False).astype(int)

# Problema 2: punto como separador de miles
columnas_enteras = ['SUSCRIPTORES_ACUEDUCTO', 'CONSUMO_ACUEDUCTO',
                    'SUSCRIPTORES_ALCANTARILLADO', 'CONSUMO_ALCANTARILLADO']
for col in columnas_enteras:
    df[col] = df[col].str.replace('.', '', regex=False).astype(int)

# Problema 3: las columnas PROMEDIO usan punto decimal, solo hay que quitar comas de miles
columnas_decimales = ['PROMEDIO_ACUEDUCTO', 'PROMEDIO_ALCANTARILLADO']
for col in columnas_decimales:
    df[col] = df[col].str.replace(',', '', regex=False).astype(float)

print(f'Filas y columnas tras limpiar: {df.shape}')
df.head()

### 1.4 Verificación obligatoria

El error de la sección 1.2 no lanza excepción, así que hay que salir a buscarlo. El consumo máximo de
acueducto en este dataset es **164.863 m3**. Si su celda imprime algo del orden de 164, la conversión
de miles salió mal.

**Qué es un `assert`.** Una afirmación que el programa comprueba: si es falsa, detiene todo con un
`AssertionError`. Sirve para escribir en el propio código las cosas que tienen que ser ciertas sí o
sí. Cuesta una línea y evita horas de análisis sobre datos rotos.

In [ ]:
print('Maximo de CONSUMO_ACUEDUCTO:', df['CONSUMO_ACUEDUCTO'].max())
print('Anios presentes:', df['ANIO'].min(), 'a', df['ANIO'].max())
print('Municipios:', df['MUNICIPIO'].nunique())
print('Estratos:', df['ESTRATO'].nunique())

assert df['CONSUMO_ACUEDUCTO'].max() > 100000, 'La conversion de miles fallo'
print('\nVerificacion superada.')

---

## 2. Paso 1 del marco: identificar

**Qué responde este paso.** Qué es la variable: de qué tipo, cuántos valores tiene de verdad y
cuántos faltan.

**Por qué va primero y no es opcional.** Porque pandas ignora los faltantes en silencio al calcular.
`.mean()` sobre una columna a la que le falta el 30% de los datos no da error ni advertencia: da el
promedio del 70% que sí está, y lo presenta como si fuera el promedio de todo. Si esa ausencia no es
aleatoria —y casi nunca lo es—, el número está sesgado y nada en la salida lo dice. El paso 1 existe
para saber eso **antes** de calcular, no después.

Los tipos de dato (`dtype`) y el `NaN` se explicaron en la clase 2, sección 3. Recordatorio de una
línea: `NaN` es la ausencia de dato, y no es lo mismo que un cero. Un consumo de 0 m3 dice que ese
grupo no consumió nada; un `NaN` dice que no sabemos cuánto consumió. Confundirlos cambia la media.

`df.info()` responde las tres preguntas del paso 1 de una sola vez.

In [ ]:
df.info()

### El paso 1 sobre la variable del cuaderno

La variable que vamos a analizar todo el cuaderno es `CONSUMO_ACUEDUCTO`: el consumo total de agua
del grupo, en metros cúbicos. El paso 1 son tres preguntas y tres líneas:

- `.dtype` — el tipo de dato. Es un atributo, va sin paréntesis.
- `.count()` — cuántos valores **no nulos** hay. Es el tamaño real de la muestra.
- `.isna().sum()` — cuántos faltan.

La celda guarda el conteo de no nulos en `no_nulos_consumo`, que es el número con el que se calcula
todo lo demás del cuaderno.

In [ ]:
variable = 'CONSUMO_ACUEDUCTO'

print('Tipo de dato:      ', df[variable].dtype)
print('Valores no nulos:  ', df[variable].count())
print('Valores faltantes: ', df[variable].isna().sum())

no_nulos_consumo = df[variable].count()

print()
print(f'Filas del DataFrame:            {len(df):,}')
print(f'Datos reales de esta variable:  {no_nulos_consumo:,}')

**Pregunta de interpretación 1.** `len(df)` y `df[variable].count()` dan el mismo número aquí, pero
son dos preguntas distintas. ¿Cuál de los dos es el que hay que poner en el denominador de un
promedio, y qué pasaría con el promedio si esta columna tuviera un 30% de faltantes y usted usara el
otro?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

El denominador correcto es `.count()`: el número de datos que existen de verdad para **esa** columna.
`len(df)` cuenta filas del DataFrame, y una fila puede existir con esa columna vacía. Aquí coinciden
porque el `dropna()` de la carga ya eliminó las filas incompletas, y por eso es un buen momento para
ver la diferencia sin que duela.

Si la columna tuviera un 30% de faltantes, `.mean()` seguiría funcionando sin error ni advertencia:
pandas ignora los `NaN` y promedia el 70% que sí está. El riesgo no es aritmético, es de
representatividad. Si lo que falta no falta al azar —y casi nunca falta al azar: suelen ser los
municipios pequeños, los meses de arranque del sistema, los grupos sin medidor— el promedio del 70%
describe una población distinta de la que usted cree estar describiendo, y nada en la salida lo
delata.

Por eso el paso 1 va antes que el paso 2. Un número calculado sobre datos cuya ausencia no se miró
es un número que no se puede defender en la sustentación.

</details>

**Pregunta de interpretación 2.** `CONSUMO_ACUEDUCTO`, ¿es una variable **cuantitativa** (se mide con
un número con el que tiene sentido hacer aritmética) o **categórica** (es una etiqueta)? Y si es
cuantitativa, ¿es **discreta** (solo toma valores separados, como un conteo) o **continua** (puede
tomar cualquier valor dentro de un rango)? ¿Y `ESTRATO`?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

`CONSUMO_ACUEDUCTO` es cuantitativa: sumarla, promediarla y compararla tiene sentido. En rigor, tal
como está guardada es **discreta**, porque el archivo la trae redondeada a metros cúbicos enteros;
pero la magnitud que representa —volumen de agua— es continua, y en la práctica se analiza como
continua porque toma miles de valores distintos. Esa es la regla útil: cuando una variable discreta
toma muchísimos valores, se trata como continua y se dibuja con histograma.

`ESTRATO` es categórica: `Estrato2` es una etiqueta. Que se escriba con un número al final no la
vuelve numérica. Sumar estratos no significa nada, y esa es la prueba definitiva. Para las
categóricas, la única de las tres medidas de centro que funciona es la **moda**.

</details>

---

## 3. Pasos 2 y 3: resumir y dispersar

### 3.1 Las tres medidas de centro

Casi todo el mundo llega a esta clase sabiendo calcular una media. Muy poca gente sabe decir **qué
es** una media, y sin eso no hay forma de elegir entre ella y la mediana con criterio. Este tramo va
a eso.

- **Media.** La suma dividida entre el número de datos.
- **Mediana.** El valor que queda justo en la mitad cuando se ordenan todos los datos.
- **Moda.** El valor que más se repite. Es la única de las tres que funciona con variables
  categóricas.

> ### Para entender qué está pasando · por qué la media se rompe y la mediana no
>
> *Si esto ya lo tiene claro, sáltese el recuadro y siga en la sección 3.2.*
>
> **La media es un punto de equilibrio.** Imagine los datos como pesas colocadas sobre una regla, cada
> una en la posición de su valor. La media es el punto exacto donde hay que poner el dedo para que la
> regla no se incline. Eso significa que **cada dato empuja con toda su magnitud**: un valor que está
> diez veces más lejos jala diez veces más fuerte. Por eso un solo dato enorme mueve la media, y por
> eso la media es tan útil cuando los datos son parecidos entre sí: usa toda la información.
>
> **La mediana es una posición, no una magnitud.** Se ordenan los datos y se señala el del medio. Cada
> dato aporta exactamente lo mismo: su lugar en la fila. Da igual si el más grande vale 300 millones o
> 300 mil millones; sigue siendo **uno** en la fila, y la fila no se mueve. Esa insensibilidad tiene un
> nombre: la mediana es un estadístico **robusto**.
>
> Ahí está la respuesta completa a "¿cuál uso?". La media resume mejor **cuando todos los datos
> merecen voz proporcional a su tamaño**. La mediana resume mejor **cuando lo que interesa es el caso
> típico** y hay valores extremos que no representan a nadie.
>
> **La analogía del salario del CEO.** Nueve empleados ganan 3 millones al mes y el gerente gana 300.
> La media es 32,7 millones: nadie en esa empresa gana eso, ni de lejos. La mediana, 3 millones, sí
> describe a un empleado. La celda de abajo lo ejecuta, y hace algo más: le duplica el sueldo al
> gerente y muestra cuál de las dos medidas se entera.

**La regla operativa de todo el semestre.** Calcule las dos y divida: **razón = media / mediana**. Si
esa razón se aparta más del 20% de 1 (o sea, si es mayor que 1,2 o menor que 0,8), la distribución
está **sesgada** y la mediana es más honesta.

**Qué es el sesgo (*skew*).** Que la distribución no es simétrica: tiene una cola más larga hacia un
lado. Si la cola larga va hacia los valores altos, se llama sesgada a la derecha, y ahí la media
queda por encima de la mediana. No es un defecto del dato: la mayoría de las variables económicas
—ingresos, precios, consumos, producción— son así, porque tienen un piso en cero y no tienen techo.

In [ ]:
# Nueve empleados con 3 millones y un gerente con 300. Cifras en millones de pesos.
salarios = pd.Series([3, 3, 3, 3, 3, 3, 3, 3, 3, 300])

print(f'Media:   {salarios.mean():6.2f} millones')
print(f'Mediana: {salarios.median():6.2f} millones')

# Ahora al gerente le duplican el sueldo. Nadie mas cambia.
salarios_2 = pd.Series([3, 3, 3, 3, 3, 3, 3, 3, 3, 600])

print()
print('El gerente pasa de 300 a 600 millones:')
print(f'Media:   {salarios.mean():6.2f}  ->  {salarios_2.mean():6.2f}   (se movio 30 millones)')
print(f'Mediana: {salarios.median():6.2f}  ->  {salarios_2.median():6.2f}   (no se movio)')
print()
print('La mediana ni se entero: el gerente sigue siendo UNO en la fila de diez.')

### 3.2 `describe()`, los ocho números de un golpe

`df[variable].describe()` entrega conteo, media, desviación estándar, mínimo, Q1, mediana, Q3 y
máximo. Cubre los pasos 2 y 3 casi completos: le falta la moda.

In [ ]:
df[variable].describe()

**Pregunta de interpretación 3.** Si `describe()` da ocho números de una sola línea, ¿para qué existe
un marco de cinco pasos? ¿Qué pregunta del marco **no** responde `describe()`?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

`describe()` cubre los pasos 2 y 3, y nada más. No cubre el paso 1 (no dice el tipo de dato ni cuántos
faltantes hay, solo cuántos válidos), no cubre el paso 4 (no dice qué **forma** tiene la distribución:
una distribución bimodal y una normal pueden dar exactamente la misma media y la misma desviación) y
no cubre el paso 5 (no decide nada sobre los outliers).

El marco existe justamente porque `describe()` no le dice si la distribución tiene dos jorobas.

</details>

### 3.3 El gancho del bloque 1, con los números reales

En el tablero quedaron dos frases sobre este mismo archivo:

- "El consumo promedio es de 5.140 m3 por registro."
- "La mitad de los registros consume menos de 501 m3."

Ninguna miente. Una es la media y la otra la mediana.

In [ ]:
media = df[variable].mean()
mediana = df[variable].median()
moda = df[variable].mode()[0]   # mode() devuelve una Series porque puede haber empate

print(f'Media:   {media:,.2f} m3')
print(f'Mediana: {mediana:,.2f} m3')
print(f'Moda:    {moda:,.2f} m3')

razon = media / mediana
print(f'\nRazon media/mediana: {razon:.2f}')

if razon > 1.2:
    print('Lectura: distribucion SESGADA A LA DERECHA. Reporte la mediana.')
elif razon < 0.8:
    print('Lectura: distribucion SESGADA A LA IZQUIERDA. Reporte la mediana.')
else:
    print('Lectura: aproximadamente simetrica. La media sirve.')

**Por qué el `[0]` de `mode()`.** Porque puede haber empate: dos valores igual de frecuentes. Para no
tener que decidir por usted, pandas devuelve **una Series** con todos los que empatan, y el `[0]` se
queda con el primero.

Note además que la moda de esta variable es 0. En una variable continua la moda casi nunca aporta
—los valores exactos rara vez se repiten—, y cuando sí se repite uno suele estar contando otra cosa.
Aquí está contando los grupos que no consumieron nada, que son más de cinco mil. Volvemos a eso en la
sección 6.

**Pregunta de interpretación 4.** La razón media/mediana da alrededor de 10. Si mañana un funcionario
le pide "el consumo promedio de agua en Caldas" para un informe público, ¿qué número le entrega y qué
le advierte?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Se entrega la **mediana**, 501 m3, y se advierte que la media de 5.140 m3 existe pero describe mal el
caso típico: está inflada por unos pocos grupos enormes (municipios grandes, estratos con miles de
suscriptores). Entregar solo la media dejaría al funcionario diciendo en público que el registro
típico consume diez veces más de lo que consume.

Lo honesto es entregar las dos y decir por qué difieren. Reportar una sola cifra de centro sin decir
cuál es y sin mirar la dispersión es la forma más común de mentir con estadística sin proponérselo.

</details>

### 3.4 Dispersión: qué tan regados están los datos

**Un número solo nunca describe una distribución.** Dos conjuntos de datos pueden tener exactamente la
misma media y no parecerse en nada: uno todo apiñado alrededor del centro y el otro desparramado. Si
solo reporta el centro, está contando la mitad de la historia, y suele ser la mitad menos interesante.

> ### Para entender qué está pasando · qué mide de verdad la desviación estándar
>
> *Si esto ya lo tiene claro, sáltese el recuadro.*
>
> **La analogía de la arquería.** Dos arqueros tiran diez flechas cada uno y los dos tienen la misma
> distancia promedio al centro. Pero el arquero A las tiene apiñadas todas juntas y el B las tiene
> desparramadas por todo el blanco. El promedio no los distingue; la consistencia sí, y eso es lo que
> mide la desviación estándar.
>
> **Qué es, en una frase:** la distancia típica de un dato a la media. Si la media del consumo es
> 5.140 y la desviación estándar es 14.706, lo que le están diciendo es "el dato corriente se aleja de
> 5.140 en unos 14.706 m3". Ese número, comparado con la media, ya le dice que los datos no se parecen
> entre sí.
>
> **Cómo se construye, y por qué cada paso.**
>
> 1. Se mide la distancia de cada dato a la media. Eso es la **desviación** de ese dato.
> 2. Se elevan al cuadrado. **Por dos razones**: primero, porque si no, las desviaciones negativas
>    cancelan exactamente a las positivas y la suma da cero siempre —es una propiedad de la media, no
>    una casualidad—; segundo, porque elevar al cuadrado **castiga más las desviaciones grandes**, y
>    eso es deliberado: estar muy lejos es mucho peor que estar un poco lejos.
> 3. Se promedian esos cuadrados. Eso es la **varianza**.
> 4. Se saca la raíz cuadrada. Eso es la **desviación estándar**.
>
> **Por qué se reporta la desviación estándar y no la varianza.** Por las unidades, y es la
> consecuencia directa del paso 2. Al elevar al cuadrado, las unidades también se elevan: la varianza
> del consumo está en metros cúbicos **al cuadrado**, que no significa nada físico. La raíz del paso 4
> existe justamente para deshacer eso y volver a metros cúbicos. Decir "la variabilidad es de 216
> millones de metros cúbicos al cuadrado" es una frase sin contenido.
>
> **Y el límite que hay que conocer:** tampoco la desviación estándar describe la forma. Dos
> distribuciones pueden compartir media **y** desviación estándar y verse completamente distintas. Lo
> comprueba la celda de abajo, y por eso el marco tiene un paso 4 que consiste en mirar el dibujo.

In [ ]:
# Dos conjuntos construidos a proposito con la MISMA media y la MISMA desviacion estandar.
# Mire las dos listas antes de mirar los numeros.
una_joroba = pd.Series([33, 45, 46, 50, 50, 50, 50, 54, 55, 67])
dos_jorobas = pd.Series([40, 41, 42, 43, 44, 56, 57, 58, 59, 60])

for nombre, serie in [('una_joroba', una_joroba), ('dos_jorobas', dos_jorobas)]:
    print(f'{nombre:12s} media = {serie.mean():5.2f}   desviacion = {serie.std():5.3f}   '
          f'mediana = {serie.median():5.2f}')
    print(f'{"":12s} {serie.tolist()}')

print()
print('Misma media, misma desviacion estandar, misma mediana. Y no se parecen en nada:')
print('la primera se apila en el centro con dos valores lejanos; la segunda son dos grupos')
print('separados sin un solo dato en el medio.')
print('Ningun numero de resumen distingue eso. El histograma si, y por eso existe el paso 4.')

**Qué son los percentiles.** El percentil `p` es el valor por debajo del cual queda el `p` por ciento
de los datos. Cuando les dieron el resultado del ICFES, el número que importaba no era el puntaje: era
el percentil. Percentil 90 no significa 90 sobre 100; significa quedar por encima del 90% de los que
presentaron. Un percentil es una **posición en la fila ordenada**, no una magnitud: por eso es
primo de la mediana y hereda su robustez.

- **Q1** = percentil 25. Por debajo está el 25% de los datos.
- **Q2** = percentil 50 = **la mediana**.
- **Q3** = percentil 75.
- **IQR** (*rango intercuartílico*) = Q3 - Q1. El rango donde vive el 50% central de los datos. Es la
  medida de dispersión que ignora los extremos: la contraparte robusta de la desviación estándar.

**Una advertencia que evita la confusión de la sección 6.** Existe la regla 68-95-99.7: en una
distribución normal, el 68% de los datos cae a una desviación estándar de la media, el 95% a dos y el
99,7% a tres. Es útil y la retomamos completa en la clase 13. Pero **vale solo si la distribución es
aproximadamente normal**, y esta no lo es ni de lejos. Por eso hoy detectamos outliers con IQR, que
no supone normalidad.

En pandas: `.std()` desviación estándar, `.var()` varianza, `.quantile(p)` el percentil con `p`
entre 0 y 1. Ojo: `0.25`, no `25`.

In [ ]:
desviacion = df[variable].std()
varianza = df[variable].var()

q1 = df[variable].quantile(0.25)
q3 = df[variable].quantile(0.75)
iqr = q3 - q1

print(f'Desviacion estandar: {desviacion:,.2f} m3')
print(f'Varianza:            {varianza:,.2f} m3 al cuadrado  <- no se reporta, no significa nada fisico')
print()
print(f'Q1 (percentil 25): {q1:,.2f} m3')
print(f'Q3 (percentil 75): {q3:,.2f} m3')
print(f'IQR (Q3 - Q1):     {iqr:,.2f} m3')
print(f'Rango: de {df[variable].min():,.0f} a {df[variable].max():,.0f} m3')

Mire el Q1: es **cero**. El 25% de los registros de este archivo consumió cero metros cúbicos. Eso no
lo dice la media por ningún lado.

### Un percentil que no viene en `describe()`

`describe()` trae los percentiles 25, 50 y 75. Los demás hay que pedirlos, y se piden como fracción:
`0.90`, no `90`. Ojo con la lectura al derecho: el percentil se define por lo que queda **debajo**, así
que "el 10% más alto" empieza donde termina el 90% de abajo.

In [ ]:
consumo_p90 = df[variable].quantile(0.90)

print(f'Percentil 90 del consumo: {consumo_p90:,.2f} m3')
print(f'Interpretacion: el 10% de los registros mas altos consume mas de {consumo_p90:,.0f} m3.')
print()
print('La escalera completa de esta variable:')
for p in [0.25, 0.50, 0.75, 0.90, 0.99]:
    print(f'  percentil {int(p * 100):3d}: {df[variable].quantile(p):>12,.2f} m3')
print(f'  maximo:        {df[variable].max():>12,.2f} m3')

**Pregunta de interpretación 5.** Mire la escalera de percentiles. De la mediana al percentil 90 el
consumo se multiplica por unas cuantas veces; del percentil 99 al máximo se vuelve a multiplicar, y
mucho más. ¿Por qué el mismo salto de percentiles cuesta cada vez más metros cúbicos, y qué le hace
eso a la media?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Porque los percentiles son **posiciones en la fila**, no magnitudes, y en esta variable la fila no
está repartida de forma pareja: la mitad de abajo se apiña entre 0 y 501 m3, y el último tramo se
estira sin techo hasta 164.863. Subir del percentil 50 al 90 cuesta pocos metros cúbicos porque ahí
los datos están pegados unos a otros; subir del 99 al 100 cuesta miles porque ahí solo hay un puñado
de registros, cada uno mucho más lejos que el anterior. Eso, dicho en una palabra, es la **cola**
derecha de la distribución.

Lo que le hace a la media es exactamente lo del recuadro del salario del CEO: como la media pesa cada
dato por su magnitud, ese último tramo la jala hacia arriba con toda su fuerza y la deja por encima
del 80% de los datos. La mediana y los percentiles no se enteran, porque cuentan puestos en la fila.

La consecuencia práctica: en una variable con cola, un solo número de centro no basta. Reportar la
mediana **y** un percentil alto (el 90, el 95) describe mucho mejor que reportar la media, porque dice
a la vez cómo es el caso típico y qué tan lejos puede llegar el caso extremo. Es lo que hace un
operador de acueducto cuando dimensiona una red: la diseña para el percentil alto, no para el
promedio.

</details>

**Pregunta de interpretación 6.** La desviación estándar es casi el triple de la media. ¿Qué le dice
eso sobre estos datos?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Que la dispersión es enorme comparada con el centro: los datos no se parecen entre sí. Cuando la
desviación estándar supera a la media, la media deja de ser un buen resumen, porque describe un
"típico" que casi ningún dato se parece.

Y hay una pista adicional: la variable no puede ser negativa (nadie consume -300 m3), pero
media menos una desviación estándar da un número negativo. Eso solo puede pasar si la distribución
está fuertemente sesgada hacia arriba, con una cola larga que estira la media. Es el mismo hallazgo de
la razón media/mediana, visto desde la dispersión.

</details>

---

## 4. GroupBy: split, apply, combine

**La analogía de los M&Ms.** Tiene una bolsa y quiere saber cuál color trae más. Los separa en
montoncitos por color (**split**), cuenta cada montoncito (**apply**), y pone los conteos lado a lado
para comparar (**combine**). Eso, exactamente eso, hace GroupBy.

**Un GroupBy son siempre tres decisiones:**

1. ¿Por cuál columna separo? — la categórica.
2. ¿Cuál columna resumo? — la numérica.
3. ¿Con cuál operación? — `.mean()`, `.sum()`, `.count()`, `.median()`.

El patrón, que se escribe igual siempre:

```python
df.groupby('COLUMNA_GRUPO')['COLUMNA_VALOR'].operacion()
```

**Por qué esto es el salto de la clase.** Pasar de "el promedio es X" a "el promedio del grupo A es X
y el del grupo B es Y". Lo segundo es infinitamente más útil: un número único describe a todo el
mundo y no describe a nadie.

> ### Para entender qué está pasando · qué es un objeto agrupado
>
> *Si esto ya lo tiene claro, sáltese el recuadro.*
>
> `df.groupby('ESTRATO')` **no calcula nada**. No devuelve una tabla: devuelve un objeto de tipo
> `DataFrameGroupBy`, que es apenas un plano de cómo quedaría partido el DataFrame. Guarda, para cada
> valor de `ESTRATO`, la lista de posiciones de fila que le corresponden. Nada más. Es el **split**, y
> es barato justamente porque no mueve los datos.
>
> El cálculo ocurre cuando usted le pide una operación: `.mean()`, `.sum()`, `.count()`, `.median()`.
> Ahí pandas recorre grupo por grupo, aplica la operación a cada uno (**apply**) y pega los resultados
> en una sola Series indexada por el nombre del grupo (**combine**). Los M&Ms, exactamente.
>
> Esto explica dos cosas que desconciertan al principio. La primera: por qué el resultado sale
> **indexado por la columna de agrupación** en vez de traerla como una columna más —el nombre del
> grupo pasó a ser la etiqueta de la fila—. La segunda: por qué imprimir `df.groupby('ESTRATO')` a
> secas no muestra ninguna tabla, sino algo como `<...DataFrameGroupBy object at 0x...>`. No está
> fallando: es que todavía no le ha pedido que calcule nada.
>
> La celda de abajo abre ese objeto por dentro: cuántos grupos hay, cómo se llaman, y cómo se saca uno
> solo con `.get_group()`.

In [ ]:
agrupado = df.groupby('ESTRATO')

print('Que devuelve groupby a secas:')
print(' ', type(agrupado).__name__)
print()
print('Cuantos grupos hay:', agrupado.ngroups)
print('Como se llaman:', list(agrupado.groups.keys()))
print()
print('Cuantas filas cayeron en cada grupo (esto ya es un apply: .size()):')
print(agrupado.size())
print()
print('Un grupo suelto, con .get_group(): las 3 primeras filas del Estrato6')
print(agrupado.get_group('Estrato6')[['MUNICIPIO', 'ANIO', 'MES',
                                      'SUSCRIPTORES_ACUEDUCTO',
                                      'CONSUMO_ACUEDUCTO']].head(3).to_string(index=False))

**Dos costumbres desde hoy.** Primera: seleccione siempre la columna que va a resumir. Segunda:
ordene el resultado con `.sort_values(ascending=False)`, porque GroupBy ordena alfabéticamente y el
hallazgo casi nunca está en orden alfabético. La celda de abajo muestra qué pasa si se saltan las dos.

In [ ]:
# Sin seleccionar la columna: promedia TODAS las numericas, incluido el anio
print('Sin seleccionar columna (fijese en la columna ANIO):')
print(df.groupby('ESTRATO').mean(numeric_only=True)[['ANIO', 'CONSUMO_ACUEDUCTO']].head(3))
print()
print('El anio promedio de un estrato no significa nada. Es ruido en la salida.')
print()
print('Sin ordenar: sale alfabetico y el hallazgo se pierde entre las filas.')
print(df.groupby('ESTRATO')['CONSUMO_ACUEDUCTO'].mean().head(4))

### 4.1 El giro

Ahora sí, bien escrito: separo por `ESTRATO`, resumo `CONSUMO_ACUEDUCTO`, con `.mean()`, y ordeno.

In [ ]:
consumo_por_estrato = (
    df.groupby('ESTRATO')['CONSUMO_ACUEDUCTO']
      .mean()
      .sort_values(ascending=False)
)

print('Consumo TOTAL promedio por estrato (m3):')
print(consumo_por_estrato)

**Alto. No ejecute la siguiente celda todavía.**

El Estrato2 encabeza y el Estrato6 va de último, con una diferencia de casi 500 veces.

**Pregunta de interpretación 7.** ¿De verdad una familia de estrato 6 gasta menos agua que una de
estrato 2? ¿Qué podría estar pasando?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

No. Lo que la tabla mide no es lo que parece. `CONSUMO_ACUEDUCTO` es el consumo **total** del grupo, y
los grupos tienen tamaños brutalmente distintos: en promedio hay 1.356 suscriptores por registro de
estrato 2 y 1,1 por registro de estrato 6. La primera tabla está midiendo, sobre todo, **cuánta gente
hay** en cada estrato.

Para responder la pregunta de verdad hay que mirar el consumo **por suscriptor**, que es lo que trae
la columna `PROMEDIO_ACUEDUCTO`. Es lo que hace la celda siguiente, y la tabla se da vuelta.

</details>

In [ ]:
promedio_por_estrato = (
    df.groupby('ESTRATO')['PROMEDIO_ACUEDUCTO']
      .mean()
      .sort_values(ascending=False)
)

print('Consumo POR SUSCRIPTOR promedio por estrato (m3):')
print(promedio_por_estrato)

In [ ]:
suscriptores_por_estrato = (
    df.groupby('ESTRATO')['SUSCRIPTORES_ACUEDUCTO']
      .mean()
      .sort_values(ascending=False)
)
print('Suscriptores promedio por registro, por estrato:')
print(suscriptores_por_estrato)

**La frase que se lleva de esta clase:** *una suma grande puede ser solo un grupo grande.* Antes de
comparar totales entre grupos, pregúntese si ya están normalizados por tamaño.

Industrial pasó de cuarto a primero, con un consumo por suscriptor casi sesenta veces mayor que el del
Estrato2. Y no es un error: son fábricas, con muy pocos suscriptores y consumos enormes cada uno.

Guarde esta cara de sorpresa. En la clase 5 esto tiene nombre propio.

### 4.2 Las otras tres agregaciones

`.mean()` ya lo vio. Los otros tres se escriben igual y responden preguntas distintas.

In [ ]:
print('SUMA: consumo acumulado por municipio, top 5')
print(df.groupby('MUNICIPIO')['CONSUMO_ACUEDUCTO'].sum().sort_values(ascending=False).head(5))
print()
print('CONTEO: cuantos registros hay por municipio, top 5')
print(df.groupby('MUNICIPIO')['CONSUMO_ACUEDUCTO'].count().sort_values(ascending=False).head(5))

### 4.3 La mediana por grupo, y el tamaño de cada grupo

La mediana por grupo responde otra pregunta que la suma por grupo: la suma dice qué municipio consume
más agua en total (y ahí gana el más grande), la mediana dice cómo es el **registro típico** de cada
municipio. Son preguntas distintas y suelen dar respuestas distintas.

Dos métodos nuevos sobre el resultado del GroupBy:

- `.idxmax()` — la **etiqueta** donde está el máximo. Aquí, el nombre del municipio.
- `.max()` — el valor máximo.

La celda calcula las dos cosas, y además cuántos registros aporta cada municipio, que es un dato que
hace falta para leer bien la tabla.

In [ ]:
mediana_por_municipio = (
    df.groupby('MUNICIPIO')['CONSUMO_ACUEDUCTO']
      .median()
      .sort_values(ascending=False)
)

print('Consumo MEDIANO por municipio, top 5 (m3):')
print(mediana_por_municipio.head(5))

top_municipio = [mediana_por_municipio.idxmax(), mediana_por_municipio.max()]
print()
print(f'Mediana mas alta: {top_municipio[0]} con {top_municipio[1]:,.2f} m3')

suma_por_municipio = (
    df.groupby('MUNICIPIO')['CONSUMO_ACUEDUCTO']
      .sum()
      .sort_values(ascending=False)
)
print(f'Suma mas alta:    {suma_por_municipio.idxmax()} con {suma_por_municipio.max():,.0f} m3')

print()
print('Cuantos registros aporta cada uno de esos dos municipios:')
conteo = df['MUNICIPIO'].value_counts()
print(f'  {top_municipio[0]}: {conteo[top_municipio[0]]:,} registros')
print(f'  {suma_por_municipio.idxmax()}: {conteo[suma_por_municipio.idxmax()]:,} registros')

**Pregunta de interpretación 8.** La mediana de un municipio con 900 registros y la de uno con 90 se
imprimen igual, con los mismos decimales y en la misma tabla. ¿Merecen la misma confianza? ¿Qué
cambiaría usted en la tabla antes de mostrársela a alguien que va a decidir con ella?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

No merecen la misma confianza. Una mediana calculada sobre 90 registros se mueve mucho más con cada
dato nuevo que una calculada sobre 900: es la misma intuición de la sopa —una cucharada de una olla
bien revuelta basta, pero una cucharada de una olla pequeña y mal revuelta no—. Con pocos datos, un
municipio puede aparecer o desaparecer del top 5 por razones que no tienen nada que ver con el
consumo real de sus habitantes.

Lo que hay que agregar a la tabla es el **tamaño del grupo**, siempre, en una columna al lado. Un
ranking de grupos sin el `n` de cada grupo es una tabla que invita a concluir de más, y es uno de los
errores que la rúbrica busca en las sustentaciones: el hallazgo espectacular que resulta venir de un
grupo con doce filas.

Y una decisión defendible cuando el desbalance es grande: fijar un umbral mínimo de registros para
entrar al ranking, decirlo en voz alta ("se excluyeron los municipios con menos de N registros") y
mostrar aparte los que quedaron fuera. Lo que no se vale es filtrarlos en silencio.

</details>

**Pregunta de interpretación 9.** El municipio con mayor consumo **total** no tiene por qué ser el de
mayor consumo **mediano**. ¿Por qué? ¿Cuál de los dos usaría para decir "aquí se gasta mucha agua"?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Porque la suma depende de cuántos registros y cuánta gente tiene el municipio, y la mediana no. Un
municipio grande puede encabezar la suma con consumos individuales perfectamente normales, solo por
tamaño; y uno pequeño con mucha industria puede tener el registro típico más alto sin acercarse al
total del grande.

"Aquí se gasta mucha agua" es ambiguo, y ahí está el punto. Si la pregunta es dónde se concentra la
demanda —para planear una obra, por ejemplo— sirve la suma. Si la pregunta es dónde se consume más por
usuario —para una campaña de ahorro— sirve la mediana o el consumo por suscriptor. Elegir la
estadística es elegir la pregunta.

</details>

---

## 5. Paso 4: visualizar

Dos gráficos, dos preguntas distintas.

- **Histograma.** ¿Qué forma tiene la distribución? Reparte los datos en cajones (*bins*) y cuenta
  cuántos caen en cada uno. Es la forma de la distribución, dibujada.
- **Boxplot** (diagrama de caja). ¿Dónde están los cuartiles y qué se sale de rango? La caja va de Q1
  a Q3 (o sea, es el IQR), la línea del medio es la mediana, los bigotes llegan hasta 1,5xIQR, y los
  puntos sueltos que quedan afuera son los outliers.

**Las cuatro formas que hay que saber reconocer:**

| Forma | Cómo se ve | Relación media-mediana | Ejemplos |
|-------|-----------|------------------------|----------|
| Normal | Campana simétrica | media = mediana | Estaturas, errores de medición |
| Sesgada a la derecha | Cola larga hacia los valores altos | media > mediana | Ingresos, precios, consumos |
| Sesgada a la izquierda | Cola larga hacia los valores bajos | media < mediana | Edad de jubilación, notas de un examen fácil |
| Bimodal | Dos jorobas | no aplica | Dos poblaciones mezcladas |

La bimodal casi siempre es una señal: falta una variable categórica que separe las dos poblaciones.

Funciones nuevas:

- `plt.subplots(1, 2)` — una figura con dos gráficos lado a lado.
- `.hist(datos, bins=n)` — histograma con n cajones.
- `.axvline(x)` — línea vertical, para marcar la media y la mediana.
- `.boxplot(datos)` — diagrama de caja.

**Regla del curso, y sale en la rúbrica:** todo gráfico lleva título, eje x, eje y y unidades. Un
gráfico sin ejes etiquetados no se puede leer sin el código al lado.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
ejes[0].hist(df[variable], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ejes[0].axvline(media, color='red', linestyle='--', linewidth=2,
                label=f'Media: {media:,.0f}')
ejes[0].axvline(mediana, color='green', linestyle='-', linewidth=2,
                label=f'Mediana: {mediana:,.0f}')
ejes[0].set_xlabel('Consumo de acueducto (m3)')
ejes[0].set_ylabel('Frecuencia (numero de registros)')
ejes[0].set_title('Distribucion del consumo de acueducto')
ejes[0].legend()

# Boxplot
ejes[1].boxplot(df[variable], patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
ejes[1].set_ylabel('Consumo de acueducto (m3)')
ejes[1].set_title('Diagrama de caja del consumo')
ejes[1].set_xticklabels(['CONSUMO_ACUEDUCTO'])

plt.tight_layout()
plt.show()

Fíjese en dos cosas:

1. La media (línea roja) está muy a la derecha de la mediana (línea verde). Eso **es** el sesgo,
   dibujado.
2. Casi todo el histograma está pegado al cero y hay una cola larguísima hacia la derecha. Esa cola
   son los grupos grandes.

### Clasificar la distribución

Esta es la parte del marco donde no hay una función que responda por usted: se mira el histograma, se
elige una de las cuatro palabras —`'normal'`, `'derecha'`, `'izquierda'`, `'bimodal'`— y la decisión
se contrasta con un número que ya está calculado, la razón media/mediana. Si el dibujo y el número no
dicen lo mismo, algo se leyó mal.

La celda deja la clasificación escrita y dibuja dos histogramas: el de siempre, con toda la variable,
y el del 95% de abajo, o sea el mismo dato con la cola recortada. Mírelos juntos antes de responder.

In [ ]:
forma_consumo = 'derecha'

print(f'Forma elegida: {forma_consumo}')
print(f'Contraste con el numero: razon media/mediana = {razon:.2f}  (media > mediana)')

corte = df[variable].quantile(0.95)
sin_cola = df[df[variable] <= corte][variable]

fig, ejes = plt.subplots(1, 2, figsize=(14, 4))

ejes[0].hist(df[variable], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ejes[0].set_title('Toda la variable')
ejes[0].set_xlabel('Consumo de acueducto (m3)')
ejes[0].set_ylabel('Frecuencia (numero de registros)')

ejes[1].hist(sin_cola, bins=50, edgecolor='black', alpha=0.7, color='darkseagreen')
ejes[1].axvline(sin_cola.mean(), color='red', linestyle='--', linewidth=2,
                label=f'Media: {sin_cola.mean():,.0f}')
ejes[1].axvline(sin_cola.median(), color='green', linestyle='-', linewidth=2,
                label=f'Mediana: {sin_cola.median():,.0f}')
ejes[1].set_title(f'Solo el 95% de abajo (consumo hasta {corte:,.0f} m3)')
ejes[1].set_xlabel('Consumo de acueducto (m3)')
ejes[1].set_ylabel('Frecuencia (numero de registros)')
ejes[1].legend()

plt.tight_layout()
plt.show()

print(f'Razon media/mediana del 95% de abajo: {sin_cola.mean() / sin_cola.median():.2f}')

**Pregunta de interpretación 10.** Al recortar el 5% más alto, el histograma cambió de escala pero no
cambió de forma: sigue apilado contra el cero y con una cola hacia la derecha, y la media sigue por
encima de la mediana. ¿Qué le dice eso sobre el origen del sesgo? ¿Es cosa de unos pocos registros
gigantes, o es otra cosa?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Que el sesgo **no** lo producen unos pocos registros gigantes: es la forma propia de la variable. Si
todo el problema fueran diez o veinte valores enormes, al quitarlos el resto tendría que verse
simétrico, y no se ve. Quitando el 5% más alto sigue apareciendo el mismo perfil, solo que con otros
números en el eje. Eso es una distribución sesgada a la derecha de verdad, no una distribución normal
contaminada por unos cuantos errores.

Tiene sentido por cómo está hecha la variable: el consumo no puede bajar de cero (hay un piso duro) y
no tiene techo, y los grupos de facturación van desde un puñado de suscriptores hasta miles. Una
variable con piso y sin techo casi siempre sale así, y por eso los ingresos, los precios, la
producción y los consumos se parecen tanto entre sí en el histograma.

La consecuencia para el resto del análisis: no hay ningún recorte que "arregle" esta variable y la
vuelva normal, así que las herramientas tienen que ser las que no suponen normalidad. Por eso los
outliers se detectan hoy con IQR y no con desviaciones estándar, y por eso se reporta la mediana. Y
por eso, cuando en la clase 13 aparezcan pruebas que sí suponen normalidad, la primera pregunta va a
ser si esta variable cumple el supuesto.

</details>

### El número de bins cambia la historia

El mismo dato, tres histogramas. Ninguno miente; los tres cuentan cosas distintas. Con 5 cajones se
pierde todo el detalle; con 300 aparece ruido que no es señal.

**Regla práctica:** empiece en 30 o 50. Suba si la forma se ve muy tosca, baje si se ve ruidosa. La
raíz del número de filas es un punto de partida, pero con 21.422 filas daría 146 cajones, que aquí es
demasiado.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4))

for eje, n_bins in zip(ejes, [5, 50, 300]):
    eje.hist(df[variable], bins=n_bins, edgecolor='black', alpha=0.7, color='steelblue')
    eje.set_title(f'bins = {n_bins}')
    eje.set_xlabel('Consumo (m3)')
    eje.set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

### Un boxplot por estrato

Lo mismo que mostraron las dos tablas de GroupBy, pero de un vistazo: `df.boxplot(column=..., by=...)`
dibuja una caja por cada valor de la columna de agrupación. `plt.suptitle('')` quita el título
automático que pandas agrega encima.

In [ ]:
df.boxplot(column='CONSUMO_ACUEDUCTO', by='ESTRATO', figsize=(14, 6), rot=45)
plt.suptitle('')
plt.title('Consumo de acueducto por estrato')
plt.xlabel('Estrato')
plt.ylabel('Consumo de acueducto (m3)')
plt.tight_layout()
plt.show()

---

## 6. Paso 5: detectar outliers con la regla 1.5xIQR

**Qué es un outlier.** Un valor que se sale del comportamiento del resto. **La analogía de la
jirafa:** un parque de perros con chihuahuas, beagles y labradores, y alguien mete una jirafa. La
jirafa sobresale porque no pertenece al grupo. La pregunta que importa no es si sobresale, es **por
qué**: ¿alguien escribió mal la especie (error de captura), o de verdad había una jirafa (caso real
inusual)?

**La regla 1.5xIQR:**

- Límite inferior = Q1 - 1,5 x IQR
- Límite superior = Q3 + 1,5 x IQR

Todo lo que quede afuera es **candidato** a outlier. Candidato, no condenado.

**Por qué IQR y no desviaciones estándar.** Porque el IQR se calcula con los cuartiles, que no se
mueven cuando hay valores extremos, mientras que la desviación estándar sí se infla con ellos: los
outliers contaminan la propia regla que debería detectarlos. Y además, la regla de las desviaciones
supone normalidad, y estos datos no son normales.

**Nunca se borran outliers por defecto.** Primero se investigan, y la decisión se documenta.

In [ ]:
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

print(f'Limite inferior: {limite_inferior:,.2f} m3')
print(f'Limite superior: {limite_superior:,.2f} m3')

outliers = df[(df[variable] < limite_inferior) | (df[variable] > limite_superior)]

print(f'\nOutliers detectados: {len(outliers):,} de {len(df):,} registros')
print(f'Porcentaje de filas: {len(outliers) / len(df) * 100:.2f}%')
print(f'Por debajo del limite inferior: {(df[variable] < limite_inferior).sum():,}')
print(f'Por encima del limite superior: {(df[variable] > limite_superior).sum():,}')

El límite inferior salió negativo, así que ningún registro puede quedar por debajo: el consumo no
puede ser menor que cero. Cuando una variable tiene un piso natural y está sesgada a la derecha, la
regla del IQR solo marca por arriba. Es esperable, no es un defecto.

### Cuánto pesan los outliers

Contar outliers dice cuántos son. No dice **cuánto pesan**, y esas dos cosas pueden ser muy distintas.
La celda calcula las dos y las imprime juntas: el porcentaje de **filas** marcadas y el porcentaje del
**consumo total** que se llevan esas filas.

In [ ]:
consumo_outliers = outliers[variable].sum()
consumo_total = df[variable].sum()
pct_consumo_outliers = consumo_outliers / consumo_total * 100

pct_filas_outliers = len(outliers) / len(df) * 100

print(f'Filas marcadas como outlier: {len(outliers):,} ({pct_filas_outliers:.2f}% de las filas)')
print(f'Consumo de esas filas:       {consumo_outliers:,.0f} m3')
print(f'Consumo de todo el dataset:  {consumo_total:,.0f} m3')
print()
print(f'Esas filas son el      {pct_filas_outliers:5.2f}% de los registros')
print(f'y se llevan el         {pct_consumo_outliers:5.2f}% del agua facturada')

**Pregunta de interpretación 11.** Un puñado de registros es una minoría de las filas y la mayoría del
consumo. Si a usted lo contratan para diseñar una campaña de ahorro de agua en Caldas, ¿a quién la
dirige, y qué habría pasado si hubiera decidido con el conteo de filas en vez de con el consumo?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

La campaña se dirige a ese grupo pequeño, porque es donde está el agua. Bajar un 5% el consumo de los
registros grandes ahorra más metros cúbicos que bajar un 30% el de la enorme mayoría de registros
pequeños. Decidir por conteo de filas —"la mayoría de los suscriptores son de estratos residenciales,
la campaña va para allá"— repartiría todo el presupuesto sobre la parte del problema que casi no mueve
la aguja.

El punto general, y es de los que se llevan a la sustentación: **contar y medir no son lo mismo**. El
conteo responde "¿cuántos casos?" y la suma responde "¿cuánto volumen?". Casi todas las variables
económicas están concentradas de esta forma, así que la pregunta "¿qué porcentaje del total se lleva
este grupo?" hay que hacerla siempre, y casi nunca coincide con "¿qué porcentaje de las filas es?".

Y el matiz honesto, que también hay que decirlo: que ahí esté el agua no significa que ahí esté el
desperdicio. Un grupo con miles de suscriptores consume mucho porque es mucha gente, no porque la
malgaste. Para hablar de desperdicio hay que mirar el consumo **por suscriptor**, que es la columna
`PROMEDIO_ACUEDUCTO` y la lección de la sección 4.

</details>

### Mírele la cara a los outliers

Contar no sirve de nada si no se investiga quiénes son. `.nlargest(n, columna)` devuelve las n filas
con el valor más alto en esa columna, con todas sus demás columnas al lado. Es la herramienta para ir
a mirar quién es la jirafa.

In [ ]:
columnas_contexto = [variable, 'SUSCRIPTORES_ACUEDUCTO', 'MUNICIPIO', 'ESTRATO', 'ANIO', 'MES']
print('Los 10 consumos mas altos del dataset:')
print(df.nlargest(10, variable)[columnas_contexto].to_string(index=False))

In [ ]:
print('Composicion de los outliers por estrato:')
print(outliers['ESTRATO'].value_counts())
print()
print('Composicion de los outliers por municipio (top 5):')
print(outliers['MUNICIPIO'].value_counts().head(5))

**Pregunta de interpretación 12.** ¿Se eliminan estos outliers? Mire de qué estratos y de qué
municipios vienen antes de responder.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

No. No son errores de captura: son los grupos con más suscriptores, de los municipios más grandes
(La Dorada, Chinchiná) y de los estratos con más hogares. Un consumo alto ahí es exactamente lo que
debe pasar. Son fábricas y ciudades, no jirafas mal escritas.

Borrarlos además destruiría el dataset: son el 11% de las filas y se llevan cerca de tres cuartas
partes del consumo total. Un análisis sin ellos ya no describiría el consumo de agua en Caldas,
describiría el consumo de agua de los municipios pequeños de Caldas, que es otra cosa y habría que
decirlo.

La respuesta profesional se escribe así: "se conservan; corresponden a grupos con muchos suscriptores
y su exclusión sesgaría el análisis. Se documenta que el 11% de los registros concentra el 75% del
consumo".

</details>

### Un dato que no es outlier pero merece nota

Hay más de cinco mil registros con consumo exactamente cero. La regla del IQR no los marca, porque el
límite inferior es negativo. Pero es un hallazgo que hay que documentar.

In [ ]:
ceros = (df[variable] == 0).sum()
print(f'Registros con consumo cero: {ceros:,} ({ceros / len(df) * 100:.1f}%)')
print()
print('De donde vienen esos ceros:')
print(df[df[variable] == 0]['ESTRATO'].value_counts())

Son combinaciones municipio-estrato-mes sin consumo registrado: en ese municipio, ese mes, ese estrato
no tuvo consumo facturado. Es un dato real, no un error. Se documenta. Si más adelante se analizan
solo los grupos activos, se filtran, **y se dice que se filtraron**.

Y note el efecto que ya tuvieron: por ellos el Q1 es cero y la moda es cero.

---

## 7. El error que no avisa (y en pandas 3, menos todavía)

Filtrar **no** modifica el original: `df[mascara]` devuelve una tabla nueva e independiente. Si usted
guarda esa tabla y le cambia un valor, está cambiando la tabla nueva. El original ni se entera.

Esto ya se dijo en la clase 2, y se repite aquí por una razón de versión: en pandas 2 este descuido
solía producir una advertencia amarilla, el `SettingWithCopyWarning`. **En pandas 3 esa advertencia ya
no existe**: el fallo es completamente silencioso. No hay error, no hay aviso, el número simplemente
no cambia.

Si lo que quiere es modificar el original, la herramienta es `.loc`, con la condición y la columna en
el mismo par de corchetes: `df.loc[condicion, 'columna'] = valor`. Se lee como una frase: "en las
filas que cumplen esto, en esta columna, ponga este valor".

La celda de abajo trabaja sobre una copia (`df.copy()`) para no dañar el `df` que venimos usando.
Fíjese en los números, no en el código.

In [ ]:
ensayo = df.copy()

# Intento 1: modificar el resultado de un filtro. No toca el original.
solo_industrial = ensayo[ensayo['ESTRATO'] == 'Industrial']
solo_industrial['CONSUMO_ACUEDUCTO'] = 0
print('Tras "modificar" el filtro, el maximo de Industrial en el original es:',
      ensayo[ensayo['ESTRATO'] == 'Industrial']['CONSUMO_ACUEDUCTO'].max())
print('Ni un error, ni una advertencia. Y el original esta intacto.')

# Intento 2: modificar el original de verdad, con .loc
ensayo.loc[ensayo['ESTRATO'] == 'Industrial', 'CONSUMO_ACUEDUCTO'] = 0
print()
print('Con .loc, el maximo de Industrial en el original es:',
      ensayo[ensayo['ESTRATO'] == 'Industrial']['CONSUMO_ACUEDUCTO'].max())
print('Y el df de verdad, sin tocar:',
      df[df['ESTRATO'] == 'Industrial']['CONSUMO_ACUEDUCTO'].max())

---

## 8. El marco de 5 pasos, completo

Todo lo anterior en un solo resumen. **Esta es la forma del entregable del reto**: no un volcado de
números, sino los cinco pasos con una lectura al final.

In [ ]:
print('=' * 62)
print(f'ANALISIS UNIVARIADO: {variable}')
print('=' * 62)

print('\n1. IDENTIFICAR')
print(f'   Tipo de dato: {df[variable].dtype}')
print(f'   Valores no nulos: {df[variable].count():,}')
print(f'   Valores faltantes: {df[variable].isna().sum():,}')

print('\n2. RESUMIR')
print(f'   Media:   {media:,.2f} m3')
print(f'   Mediana: {mediana:,.2f} m3')
print(f'   Moda:    {moda:,.2f} m3')
print(f'   Razon media/mediana: {razon:.2f}')

print('\n3. DISPERSAR')
print(f'   Desviacion estandar: {desviacion:,.2f} m3')
print(f'   IQR: {iqr:,.2f} m3   (Q1 = {q1:,.2f}, Q3 = {q3:,.2f})')
print(f'   Rango: {df[variable].min():,.0f} a {df[variable].max():,.0f} m3')

print('\n4. VISUALIZAR')
print('   Forma: sesgada a la derecha (media muy por encima de la mediana)')

print('\n5. DETECTAR')
print(f'   Limites 1.5xIQR: [{limite_inferior:,.2f}, {limite_superior:,.2f}]')
print(f'   Outliers: {len(outliers):,} ({len(outliers) / len(df) * 100:.2f}% de las filas)')
print('   Decision: se conservan, corresponden a grupos con muchos suscriptores')

print('\n' + '=' * 62)

**Cierre.** Escriba una frase, una sola, que resuma qué aprendió sobre el consumo de agua en Caldas.
**La frase no puede contener ningún número.**

Ejemplo de la forma que debe tener, no lo copie: "La mayoría de los grupos de facturación consume
poco, y el promedio general está inflado por unos pocos grupos muy grandes, así que reportar el
promedio sin la mediana da una idea equivocada del consumo típico."

*Tu respuesta:*

---

## 9. Punto de control

Este cuaderno no se autocalifica: no hay nada que teclear en él. El punto de control es usted
respondiéndose, sin abrir los desplegables, estas tres preguntas:

1. ¿Sabría decidir entre media y mediana para una variable nueva, y defender la elección con un
   número?
2. ¿Sabría escribir un GroupBy desde cero y explicar por qué el promedio por grupo puede contradecir
   al promedio general?
3. ¿Sabría calcular los límites 1.5xIQR, contar los outliers y argumentar si se conservan o no?

Si alguna respuesta es "no", no pase de largo: el reto arranca dando por sabido todo lo de este
cuaderno, y es donde sí hay que escribir código. Levante la mano ahora, que el profesor está en el
salón.

---

## 10. Preguntas que siempre salen

**¿Cuándo uso media y cuándo mediana?** Calcule las dos. Si la razón media/mediana se aparta más del
20% de 1, la distribución está sesgada y la mediana es más honesta. Para ingresos, precios y consumos,
casi siempre mediana.

**¿Por qué se elevan al cuadrado las desviaciones en la varianza?** Dos razones: para que las
negativas no cancelen a las positivas, y para castigar más las desviaciones grandes. La raíz cuadrada
al final devuelve el resultado a las unidades originales.

**¿Y si mi variable no tiene ningún outlier?** Perfecto, es un hallazgo. Se documenta como "no se
detectaron outliers con el método IQR". No todas las variables tienen jirafas.

**¿Puedo hacer GroupBy por dos columnas a la vez?** Sí, y es muy útil, pero hoy no. Hoy el objetivo es
que el patrón de tres decisiones quede automático. Agrupar por varias columnas y `.agg()` con
diccionarios llegan más adelante.

**¿La moda sirve para algo en variables numéricas continuas?** Poco: en continuas casi nunca se repite
un valor exacto. Donde brilla es en categóricas: el estrato más frecuente, el municipio más frecuente.
Ahí es la única de las tres medidas de centro que funciona.

**¿Qué pasa si borro los outliers y los resultados se ven más bonitos?** Que su análisis ya no
describe la realidad. En este dataset los outliers son las fábricas y las ciudades grandes, y ambas
existen. Bonito no es correcto.

**¿La regla 68-95-99.7 aplica a estos datos?** No. Vale para distribuciones aproximadamente normales,
y esta tiene razón media/mediana de 10. Por eso hoy se detectan outliers con IQR, que no supone
normalidad. La regla completa vuelve en la clase 13.

**¿Este dataset sirve para el proyecto de mi equipo?** No. Los CSV de las clases son material de
enseñanza, elegidos por lo que permiten enseñar. El dataset del proyecto lo consigue cada equipo, y
tiene que poder decir de dónde salió y bajo qué condiciones lo usa. Los criterios están en la guía de
entrega del Momento 1.

---

## Resumen

| Lo que hizo | Con qué |
|-------------|---------|
| Cargar sin que pandas adivine | `pd.read_csv(ruta, dtype=str)` |
| Quitar separadores y convertir | `.str.replace(...)`, `.astype(int)`, `.astype(float)` |
| Paso 1: identificar | `df.info()`, `.dtype`, `.count()`, `.isna().sum()` |
| Pasos 2 y 3, de un golpe | `df[col].describe()` |
| Centro | `.mean()`, `.median()`, `.mode()[0]` |
| Dispersión | `.std()`, `.var()`, `.quantile(p)` |
| GroupBy | `df.groupby('GRUPO')['VALOR'].mean()` y `.sort_values(ascending=False)` |
| Máximo de una Series y su etiqueta | `.max()`, `.idxmax()` |
| Paso 4: visualizar | `plt.hist(bins=n)`, `plt.boxplot()`, `.axvline()` |
| Paso 5: detectar | `q1 - 1.5*iqr`, `q3 + 1.5*iqr` |
| Mirarle la cara a los outliers | `df.nlargest(n, columna)` |
| Modificar el original | `df.loc[condicion, 'columna'] = valor` |

**Las cinco reglas que no se negocian:**

1. Antes de reportar cualquier media, calcule la razón media/mediana.
2. La varianza está en unidades al cuadrado. Se reporta la desviación estándar.
3. GroupBy son tres decisiones, y siempre `.sort_values()`.
4. Una suma grande puede ser solo un grupo grande.
5. Los outliers se investigan antes de tocarlos.

**Autoevaluación honesta.** Si puede responder que sí a estas cinco, está listo para el reto:

- [ ] Puedo decir qué tipo tiene una variable, cuántos no nulos y cuántos faltantes.
- [ ] Sé elegir entre media y mediana y justificar la elección con un número.
- [ ] Sé interpretar una desviación estándar y un percentil en una frase en español.
- [ ] Escribo un GroupBy sin plantilla y sé explicar el split, el apply y el combine.
- [ ] Calculo los límites 1.5xIQR, cuento los outliers y argumento si se eliminan.

**Ahora:** el bloque 3. `../reto/README.md` y `../reto/reto_starter.ipynb`, con 20.000 filas de
evaluaciones agropecuarias. El mismo marco, tres veces, sobre datos que no vio aquí.